# 11 · Tool use, structured output and agents

The model never *executes* anything. It emits a **tool call** (name + JSON arguments); your code validates and runs it, returns the result as a `tool` message, and the model continues. An **agent** is this in a loop with a goal and limits.

**What you will learn**
- Describing tools with JSON schemas; validating arguments *before* execution
- The agent loop: steps, observations, error feedback, max-steps, human confirmation gates
- Structured output: robust JSON parsing + schema validation + retry (and when to use constrained decoding instead)
- Evaluation: tool-selection accuracy, argument correctness, task success, steps and cost per task

Runs offline with `MockLLM`; swap in `OpenAICompatibleClient` (vLLM with `--enable-auto-tool-choice`) for a real model.

**Source:** `llm_edu_suite/agents.py`, `llm_edu_suite/llm_client.py`.

In [1]:
import os, sys, math, time, random, json
ROOT = os.path.abspath(".")
while not os.path.isdir(os.path.join(ROOT, "llm_edu_suite")) and os.path.dirname(ROOT) != ROOT:
    ROOT = os.path.dirname(ROOT)                    # climb from notebooks/<topic>/ up to the repo root
sys.path.insert(0, ROOT)                            # so that `import llm_edu_suite` works from any notebook
import numpy as np
random.seed(0); np.random.seed(0)

In [2]:
from llm_edu_suite.agents import ToolRegistry, Agent, validate_args, parse_json_output, structured_output
from llm_edu_suite.llm_client import MockLLM
from llm_edu_suite.rag import chunk_documents, Retriever

registry = ToolRegistry()

@registry.register("calculate", "Evaluate an arithmetic expression and return the numeric result",
                   {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]})
def calculate(expression: str) -> str:
    if not all(ch in "0123456789+-*/(). " for ch in expression):        # a real sandbox would be much stricter than this
        raise ValueError("only arithmetic is allowed")
    return str(eval(expression, {"__builtins__": {}}, {}))

docs = {"policy.md": "Refunds are accepted within 30 days of purchase with a receipt. Shipping costs are not refunded. Digital products cannot be refunded once downloaded.",
        "hours.md": "Support is available Monday to Friday from 9am to 6pm. Weekend requests are answered on the next business day."}
retriever = Retriever(chunk_documents(docs, size=20, overlap=5))

@registry.register("search_docs", "Search the company documentation knowledge base for policies, hours and procedures",
                   {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]})
def search_docs(query: str) -> str:
    return " | ".join(c.text for c in retriever.retrieve(query, k=2))

@registry.register("get_weather", "Get the current temperature for a city",
                   {"type": "object", "properties": {"city": {"type": "string"}, "unit": {"type": "string", "enum": ["C", "F"]}}, "required": ["city"]})
def get_weather(city: str, unit: str = "C") -> dict:
    return {"city": city, "temp": 19 if unit == "C" else 66, "unit": unit, "condition": "cloudy"}

print(json.dumps(registry.to_openai()[0], indent=1))     # this is what the model sees

{
 "type": "function",
 "function": {
  "name": "calculate",
  "description": "Evaluate an arithmetic expression and return the numeric result",
  "parameters": {
   "type": "object",
   "properties": {
    "expression": {
     "type": "string"
    }
   },
   "required": [
    "expression"
   ]
  }
 }
}


## 1. Validation before execution
Never trust generated arguments blindly: validate types/required/enums, and return errors **as text** so the model can correct itself instead of crashing the loop.

In [3]:
schema = registry.tools["get_weather"].parameters
print(validate_args({"city": "Lima"}, schema))                       # []
print(validate_args({"city": 5, "unit": "K"}, schema))                # wrong type and a value outside the enum
print(validate_args({"unit": "C"}, schema))                          # a required argument is missing
print(registry.execute("calculate", {"expression": "import os"}))    # the error comes back as a string instead of raising
print(registry.execute("nope", {}))

[]
["argument 'city' should be string, got int", "argument 'unit' must be one of ['C', 'F']"]
["missing required argument 'city'"]
ERROR: ValueError: only arithmetic is allowed
ERROR: unknown tool 'nope'. Available: ['calculate', 'search_docs', 'get_weather']


## 2. The agent loop
`Agent.run` alternates model ↔ tools until the model answers in text or `max_steps` is hit. Observations are truncated (`max_obs_chars`) so the context does not explode. A `confirm` callback gates dangerous actions (human-in-the-loop).

In [4]:
llm = MockLLM()
agent = Agent(llm, registry, max_steps=4)
for task in ["Please calculate (17 * 23) + 4 and give me the arithmetic result",
             "search the documentation: can I get a refund after 30 days?",
             "What is the weather temperature in Lima?"]:
    run = agent.run(task, verbose=True)
    print(f"USER: {task}\nAGENT: {run.answer}\n   ({len(run.steps)} steps, stopped: {run.stopped_reason})\n")

  [step 0] calculate({"expression": "(17 * 23) + 4"}) -> 395
USER: Please calculate (17 * 23) + 4 and give me the arithmetic result
AGENT: Based on the tool result: 395
   (2 steps, stopped: answer)

  [step 0] search_docs({"query": "search the documentation: can I get a refund after 30 days?"}) -> Refunds are accepted within 30 days of purchase with a receipt. Shipping costs are not refunded. Digital products cannot
USER: search the documentation: can I get a refund after 30 days?
AGENT: Based on the tool result: Refunds are accepted within 30 days of purchase with a receipt. Shipping costs are not refunded. Digital products cannot be | Support is available Monday to Friday from 9am to 6pm. Weekend requests are answered on the next business day.
   (2 steps, stopped: answer)

  [step 0] get_weather({"city": "Lima", "unit": "C"}) -> {"city": "Lima", "temp": 19, "unit": "C", "condition": "cloudy"}
USER: What is the weather temperature in Lima?
AGENT: Based on the tool result: {"city": "

In [5]:
# A human confirmation gate: only the read-only tools are approved, everything else is denied.
guarded = Agent(llm, registry, confirm=lambda name, args: name in ("search_docs", "get_weather"))
run = guarded.run("Please calculate 2 + 2 arithmetic", verbose=True)
print("AGENT:", run.answer)

  [step 0] calculate({"expression": "2 + 2"}) -> ERROR: action not confirmed by the user
AGENT: Based on the tool result: ERROR: action not confirmed by the user


## 3. Structured output
Asking for JSON in the prompt fails some percent of the time. Two robust options: (a) **constrained decoding** on the server (`response_format={"type":"json_schema", ...}` in vLLM/SGLang via XGrammar/Outlines) — guaranteed valid, free; (b) client-side **parse → validate → retry with the error**, implemented in `structured_output`.

In [6]:
print(parse_json_output('Sure! Here it is:\n```json\n{"a": 1, "b": [1, 2]}\n```\nLet me know.'))
print(parse_json_output('{"nested": {"x": 1}} trailing text'))

schema = {"type": "object", "properties": {"email": {"type": "string"}, "amount": {"type": "number"}, "id": {"type": "string"}},
          "required": ["email", "amount"]}
text = "Order #B77-120 from Carlos Ruiz, total $2,480.00, contact carlos.ruiz@example.com, 3 units"
print(structured_output(llm, f"Extract the fields.\n\nText:\n{text}", schema))

{'a': 1, 'b': [1, 2]}
{'nested': {'x': 1}}
{'email': 'carlos.ruiz@example.com', 'amount': 2480.0, 'id': 'B77-120'}


## 4. Evaluation

Agents are evaluated on **trajectories**, not just final answers:
1. **Tool-selection accuracy** — did it call the right tool (or none) for each test input?
2. **Argument correctness** — schema-valid and semantically right?
3. **Task success** — final answer correct (exact match / judge).
4. **Efficiency** — steps and tokens per task; **safety** — no unconfirmed dangerous calls.

Build a test set of (input → expected tool, expected answer) and run it after every prompt/tool-description change: tool descriptions are the agent's "API docs" and small wording changes move these numbers.

In [7]:
test_cases = [
    ("Please calculate 12 * 12 arithmetic", "calculate", "144"),
    ("Compute the arithmetic expression (3 + 5) * 2", "calculate", "16"),
    ("search the documentation: what are the support hours?", "search_docs", "Monday to Friday"),
    ("search the documentation for the refund policy on digital products", "search_docs", "cannot be refunded"),
    ("weather temperature in Cusco", "get_weather", "19"),
]
tool_ok = arg_ok = ans_ok = 0; total_steps = 0
for text, exp_tool, exp_substr in test_cases:
    run = agent.run(text)
    called = [s.tool for s in run.steps if s.tool]
    tool_ok += bool(called) and called[0] == exp_tool
    arg_ok += bool(called) and not str(run.steps[0].result).startswith("ERROR")
    ans_ok += exp_substr.lower() in run.answer.lower()
    total_steps += len(run.steps)
n = len(test_cases)
print(f"tool selection {tool_ok}/{n} | valid arguments {arg_ok}/{n} | answer contains expected {ans_ok}/{n} | avg steps {total_steps/n:.1f}")
print(f"token usage: {llm.usage.calls} calls, {llm.usage.prompt_tokens} prompt tokens, est. cost ${llm.usage.cost():.5f}")

tool selection 5/5 | valid arguments 5/5 | answer contains expected 5/5 | avg steps 2.0
token usage: 19 calls, 244 prompt tokens, est. cost $0.00050


**Production notes:** use the model's native tool-calling format via the chat template; sandbox code execution (containers, timeouts, allow-lists); log every trajectory; cap steps and spend; put retrieved/tool text in delimiters and tell the model it is *data* (see notebook 13 on prompt injection). The Model Context Protocol (MCP) standardizes how tools are exposed to models.

**Next:** `12_recommender_systems.ipynb`.